## Dataset Playground 

In [ ]:
import os, json, textwrap
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, get_dataset_config_names

sns.set_theme(style="whitegrid", font_scale=0.9)
FIG_DIR = "assets/figs"
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    name = os.path.splitext(name)[0] + ".pdf"
    plt.savefig(os.path.join(FIG_DIR, name), bbox_inches="tight", dpi=300)

def parse_qa(qa):
    if isinstance(qa, dict):
        return qa
    if isinstance(qa, str):
        try:
            return json.loads(qa)
        except Exception:
            return {}
    return {}

In [ ]:
dataset_names = get_dataset_config_names("Manhph2211/PulseLM")
print(f"Configs ({len(dataset_names)}): {dataset_names}")

ALL_SPLITS = ["train", "validation", "test"]
raw = {name: {split: load_dataset("Manhph2211/PulseLM", name, split=split) for split in ALL_SPLITS} for name in dataset_names}

In [ ]:
SPLIT = "test"

In [ ]:
split_rows = []
for name in dataset_names:
    row = {"dataset": name}
    for split in ALL_SPLITS:
        row[split] = len(raw[name][split])
    row["total"] = sum(row[s] for s in ALL_SPLITS)
    split_rows.append(row)

df_splits = pd.DataFrame(split_rows).set_index("dataset")
df_splits.loc["TOTAL"] = df_splits.sum()
print(df_splits.to_string())

In [ ]:
per_dataset_cats = defaultdict(Counter)
per_cat_answers = defaultdict(Counter)
per_dataset_per_cat_answers = defaultdict(lambda: defaultdict(Counter))
qa_counts_per_example = defaultdict(list)

SKIP_CATS = {'activity_label'} 

_ANSWER_REMAP = {
    "sqi_category": {"good_quality": "good_quality", "noisy_or_distorted": "noisy", "symmetric_unusual": "noisy"},
    "spo2_category": {"normal": "normal", "mild_hypoxemia": "abnormal", "moderate_hypoxemia": "abnormal", "severe_hypoxemia": "abnormal"},
}

for name in dataset_names:
    ds = raw[name][SPLIT]
    for i in range(len(ds)):
        qa = parse_qa(ds[i]["qa"])
        qa_counts_per_example[name].append(len(qa))
        for cat, payload in qa.items():
            if cat in SKIP_CATS:
                continue
            ans = payload.get("answer", str(payload)) if isinstance(payload, dict) else str(payload)
            ans = _ANSWER_REMAP.get(cat, {}).get(ans, ans)
            per_dataset_cats[name][cat] += 1
            per_cat_answers[cat][ans] += 1
            per_dataset_per_cat_answers[name][cat][ans] += 1

all_cats = sorted(per_cat_answers.keys())
print(f"Total QA categories: {len(all_cats)}")
for cat in all_cats:
    total = sum(per_cat_answers[cat].values())
    print(f"  {cat}: {total:,} examples  answers={dict(per_cat_answers[cat])}")

In [ ]:
heatmap_data = pd.DataFrame(
    {name: {cat: per_dataset_cats[name].get(cat, 0) for cat in all_cats} for name in dataset_names}
).T

fig, axes = plt.subplots(1, 2, figsize=(24, 6))

sns.heatmap(
    (heatmap_data > 0).astype(int), ax=axes[0],
    cmap="Greys", linewidths=0.5, linecolor="#cccccc",
    annot=True, fmt="d", cbar_kws={"label": "present"},
    annot_kws={"fontsize": 7},
)
axes[0].tick_params(axis="x", rotation=40, labelsize=7)
axes[0].tick_params(axis="y", rotation=0, labelsize=8)

heatmap_norm = heatmap_data.div(heatmap_data.sum(axis=1).replace(0, 1), axis=0)
sns.heatmap(
    heatmap_norm, ax=axes[1],
    cmap="Greys", linewidths=0.5, linecolor="#cccccc",
    fmt=".2f", annot=True, cbar_kws={"label": "fraction of QA pairs"},
    annot_kws={"fontsize": 7},
)
axes[1].tick_params(axis="x", rotation=40, labelsize=7)
axes[1].tick_params(axis="y", rotation=0, labelsize=8)

plt.tight_layout()
savefig("category_heatmap.pdf")
plt.show()

In [ ]:
CAT_DISPLAY = {
    'heart_rate_category':     'Heart Rate',
    'blood_pressure_category': 'Blood Pressure',
    'hrv_sdnn_category':       'HRV-SDNN',
    'hrv_rmssd_category':      'HRV-RMSSD',
    'hrv_pnn50_category':      'HRV-pNN50',
    'af_label':                'AF Detection',
    'arrhythmia_category':     'Arrhythmia',
    'spo2_category':           'SpO2',
    'rr_category':             'Respiratory Rate',
    'sqi_category':            'Signal Quality',
    'stress_label':            'Stress',
    'sdb_label':               'Sleep Apnea',
}

global_counts = pd.Series(
    {cat: sum(per_dataset_cats[n].get(cat, 0) for n in dataset_names) for cat in all_cats}
).sort_values(ascending=False)

n = len(global_counts)
gray_vals = ["#%02X%02X%02X" % tuple([int(80 + 120 * i / max(n - 1, 1))] * 3) for i in range(n)]

fig, ax = plt.subplots(figsize=(13, 4))
bars = ax.bar(global_counts.index, global_counts.values,
              color=gray_vals, edgecolor="white", linewidth=0.3)
ax.set_ylabel("count", fontsize=10)
ax.tick_params(labelsize=8)
ax.ticklabel_format(axis="y", style="scientific", scilimits=(0, 3))
ax.yaxis.get_offset_text().set_fontsize(10)
ax.set_xticks(range(len(global_counts)))
ax.set_xticklabels([CAT_DISPLAY.get(c, c) for c in global_counts.index], rotation=35, ha="right", fontsize=9)
for bar, val in zip(bars, global_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(global_counts) * 0.005,
            f"{val:,}", ha="center", va="bottom", fontsize=9, color="#555555")
plt.tight_layout()
savefig("global_category_bar.pdf")
plt.show()

In [ ]:
ncols = 4
nrows = int(np.ceil(len(all_cats) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 3.5))

for ax, cat in zip(axes.flat, all_cats):
    counts = per_cat_answers[cat]
    labels, values = zip(*sorted(counts.items(), key=lambda x: -x[1]))
    n_bars = len(labels)
    gray_vals = ["#%02X%02X%02X" % tuple([int(80 + 120 * i / max(n_bars - 1, 1))] * 3) for i in range(n_bars)]
    bars = ax.barh(list(labels), list(values), color=gray_vals, edgecolor="white", linewidth=0.3)
    ax.set_title(cat, fontsize=9, fontweight="bold")
    ax.set_xlabel("count", fontsize=7)
    ax.tick_params(labelsize=9)
    ax.ticklabel_format(axis="x", style="scientific", scilimits=(0, 3))
    ax.xaxis.get_offset_text().set_fontsize(6)
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + max(values) * 0.01, bar.get_y() + bar.get_height() / 2,
                f"{val:,}", va="center", fontsize=6, color="#555555")

for ax in axes.flat[len(all_cats):]:
    ax.set_visible(False)

plt.tight_layout()
savefig("answer_dist_per_category.pdf")
plt.show()

In [ ]:
_all_cat_answers = defaultdict(Counter)

_ANSWER_REMAP = {
    "sqi_category": {
        "good_quality": "good_quality",
        "noisy_or_distorted": "noisy",
        "symmetric_unusual": "noisy",
    },
    "spo2_category": {
        "normal": "normal",
        "mild_hypoxemia": "abnormal",
        "moderate_hypoxemia": "abnormal",
        "severe_hypoxemia": "abnormal",
    },
}

for _name in dataset_names:
    for _split in ALL_SPLITS: #["test"]:
        _ds = raw[_name][_split]
        for _i in range(len(_ds)):
            _qa = parse_qa(_ds[_i]['qa'])
            for _cat, _payload in _qa.items():
                _ans = _payload.get('answer', str(_payload)) if isinstance(_payload, dict) else str(_payload)
                _ans = _ANSWER_REMAP.get(_cat, {}).get(_ans, _ans)
                _all_cat_answers[_cat][_ans] += 1

ANSWER_DISPLAY = {
    'bradycardia': 'Brady.', 'normal': 'Normal', 'tachycardia': 'Tachy.',
    'elevated': 'Elevated', 'hypertension_stage1': 'HTN-1',
    'hypertension_stage2': 'HTN-2', 'hypertensive_crisis': 'Crisis',
    'low': 'Low', 'high': 'High',
    'af': 'AF', 'non_af': 'Non-AF',
    'sinus_rhythm': 'Sinus', 'pvc': 'PVC', 'pac': 'PAC', 'vt': 'VT', 'svt': 'SVT',
    'unknown_0': 'Unknown',
    'good_quality': 'Good', 'noisy': 'Noisy',
    'bradypnea': 'Brady.', 'tachypnea': 'Tachy.',
    'abnormal': 'Abnormal',
    'normal_ahi<5': 'Normal', 'mild_5<=ahi<15': 'Mild',
    'moderate_15<=ahi<30': 'Moderate', 'severe_ahi>=30': 'Severe',
    'baseline': 'Baseline', 'stress': 'Stress', 'amusement': 'Amusement', 'meditation': 'Meditation',
}

CAT_PALETTES = {
    'heart_rate_category':     ['#666666', '#999999', '#CCCCCC'],
    'blood_pressure_category': ['#444444', '#666666', '#888888', '#AAAAAA', '#CCCCCC'],
    'hrv_sdnn_category':       ['#666666', '#999999', '#CCCCCC'],
    'hrv_rmssd_category':      ['#666666', '#999999', '#CCCCCC'],
    'hrv_pnn50_category':      ['#666666', '#999999', '#CCCCCC'],
    'af_label':                ['#777777', '#BBBBBB'],
    'arrhythmia_category':     ['#444444', '#666666', '#888888', '#AAAAAA', '#BBBBBB', '#CCCCCC'],
    'spo2_category':           ['#777777', '#BBBBBB'],
    'rr_category':             ['#666666', '#999999', '#CCCCCC'],
    'sqi_category':            ['#777777', '#BBBBBB'],
    'stress_label':            ['#555555', '#777777', '#999999', '#BBBBBB'],
    'sdb_label':               ['#555555', '#777777', '#999999', '#BBBBBB'],
}

LABEL_ORDER = {
    'heart_rate_category':     ['bradycardia', 'normal', 'tachycardia'],
    'blood_pressure_category': ['normal', 'elevated', 'hypertension_stage1', 'hypertension_stage2', 'hypertensive_crisis'],
    'hrv_sdnn_category':       ['low', 'normal', 'high'],
    'hrv_rmssd_category':      ['low', 'normal', 'high'],
    'hrv_pnn50_category':      ['low', 'normal', 'high'],
    'af_label':                ['non_af', 'af'],
    'arrhythmia_category':     ['sinus_rhythm', 'pvc', 'pac', 'vt', 'svt', 'unknown_0'],
    'spo2_category':           ['normal', 'abnormal'],
    'rr_category':             ['bradypnea', 'normal', 'tachypnea'],
    'sqi_category':            ['good_quality', 'noisy'],
    'stress_label':            ['baseline', 'stress', 'amusement', 'meditation'],
    'sdb_label':               ['normal_ahi<5', 'mild_5<=ahi<15', 'moderate_15<=ahi<30', 'severe_ahi>=30'],
}

cat_order_fig = [
    'heart_rate_category', 'blood_pressure_category',
    'hrv_sdnn_category', 'hrv_rmssd_category', 'hrv_pnn50_category',
    'af_label', 'arrhythmia_category',
    'spo2_category', 'rr_category', 'sqi_category',
    'stress_label', 'sdb_label',
] 

cat_display_fig = {
    'heart_rate_category':     'Heart Rate',
    'blood_pressure_category': 'Blood Pressure',
    'hrv_sdnn_category':       'HRV-SDNN',
    'hrv_rmssd_category':      'HRV-RMSSD',
    'hrv_pnn50_category':      'HRV-pNN50',
    'af_label':                'AF Detection',
    'arrhythmia_category':     'Arrhythmia',
    'spo2_category':           'SpO2',
    'rr_category':             'Respiratory Rate',
    'sqi_category':            'Signal Quality',
    'stress_label':            'Stress',
    'sdb_label':               'Sleep Apnea',
}

cat_order_fig = [c for c in cat_order_fig if c in _all_cat_answers]

fig, axes = plt.subplots(3, 4, figsize=(10, 7))
axes_flat = axes.flatten()

for idx, cat in enumerate(cat_order_fig):
    ax = axes_flat[idx]
    counter = _all_cat_answers[cat]

    labels_ordered = [l for l in LABEL_ORDER.get(cat, []) if l in counter]
    labels_ordered += [l for l in sorted(counter.keys()) if l not in labels_ordered]

    counts = [counter[l] for l in labels_ordered]
    display_labels = [ANSWER_DISPLAY.get(l, l) for l in labels_ordered]
    colors = CAT_PALETTES.get(cat, ['#B0B0B0'] * len(labels_ordered))
    if len(colors) < len(labels_ordered):
        colors = colors + ['#D0D0D0'] * (len(labels_ordered) - len(colors))

    bars = ax.bar(range(len(labels_ordered)), counts,
                  color=colors[:len(labels_ordered)], edgecolor='white', linewidth=0.3)

    total = sum(counts)
    for bar, count in zip(bars, counts):
        pct = count / total * 100
        y_off = bar.get_height() + total * 0.01
        ax.text(bar.get_x() + bar.get_width() / 2, y_off,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=6, color='#555555')

    ax.set_xticks(range(len(labels_ordered)))
    ax.set_xticklabels(display_labels, rotation=40, ha='right', fontsize=6.5)
    ax.set_title(cat_display_fig[cat], fontsize=9, fontweight='bold', pad=4)
    ax.set_ylabel('Count', fontsize=6.5)
    ax.tick_params(labelsize=7)
    ax.ticklabel_format(axis='y', style='scientific', scilimits=(0, 3))
    ax.yaxis.get_offset_text().set_fontsize(6)

for ax in axes_flat[len(cat_order_fig):]:
    ax.set_visible(False)

plt.tight_layout(h_pad=1.5, w_pad=1.0)
savefig('label_distribution_in_all_sets.pdf')
plt.show()
print('Saved: assets/figs/label_distribution_in_all_sets.pdf')

In [ ]:
_all_cat_answers = defaultdict(Counter)

_ANSWER_REMAP = {
    "sqi_category": {
        "good_quality": "good_quality",
        "noisy_or_distorted": "noisy",
        "symmetric_unusual": "noisy",
    },
    "spo2_category": {
        "normal": "normal",
        "mild_hypoxemia": "abnormal",
        "moderate_hypoxemia": "abnormal",
        "severe_hypoxemia": "abnormal",
    },
}

for _name in dataset_names:
    for _split in ["test"]:
        _ds = raw[_name][_split]
        for _i in range(len(_ds)):
            _qa = parse_qa(_ds[_i]['qa'])
            for _cat, _payload in _qa.items():
                _ans = _payload.get('answer', str(_payload)) if isinstance(_payload, dict) else str(_payload)
                _ans = _ANSWER_REMAP.get(_cat, {}).get(_ans, _ans)
                _all_cat_answers[_cat][_ans] += 1

ANSWER_DISPLAY = {
    'bradycardia': 'Brady.', 'normal': 'Normal', 'tachycardia': 'Tachy.',
    'elevated': 'Elevated', 'hypertension_stage1': 'HTN-1',
    'hypertension_stage2': 'HTN-2', 'hypertensive_crisis': 'Crisis',
    'low': 'Low', 'high': 'High',
    'af': 'AF', 'non_af': 'Non-AF',
    'sinus_rhythm': 'Sinus', 'pvc': 'PVC', 'pac': 'PAC', 'vt': 'VT', 'svt': 'SVT',
    'unknown_0': 'Unknown',
    'good_quality': 'Good', 'noisy': 'Noisy',
    'bradypnea': 'Brady.', 'tachypnea': 'Tachy.',
    'abnormal': 'Abnormal',
    'normal_ahi<5': 'Normal', 'mild_5<=ahi<15': 'Mild',
    'moderate_15<=ahi<30': 'Moderate', 'severe_ahi>=30': 'Severe',
    'baseline': 'Baseline', 'stress': 'Stress', 'amusement': 'Amusement', 'meditation': 'Meditation',
}

CAT_PALETTES = {
    'heart_rate_category':     ['#666666', '#999999', '#CCCCCC'],
    'blood_pressure_category': ['#444444', '#666666', '#888888', '#AAAAAA', '#CCCCCC'],
    'hrv_sdnn_category':       ['#666666', '#999999', '#CCCCCC'],
    'hrv_rmssd_category':      ['#666666', '#999999', '#CCCCCC'],
    'hrv_pnn50_category':      ['#666666', '#999999', '#CCCCCC'],
    'af_label':                ['#777777', '#BBBBBB'],
    'arrhythmia_category':     ['#444444', '#666666', '#888888', '#AAAAAA', '#BBBBBB', '#CCCCCC'],
    'spo2_category':           ['#777777', '#BBBBBB'],
    'rr_category':             ['#666666', '#999999', '#CCCCCC'],
    'sqi_category':            ['#777777', '#BBBBBB'],
    'stress_label':            ['#555555', '#777777', '#999999', '#BBBBBB'],
    'sdb_label':               ['#555555', '#777777', '#999999', '#BBBBBB'],
}

LABEL_ORDER = {
    'heart_rate_category':     ['bradycardia', 'normal', 'tachycardia'],
    'blood_pressure_category': ['normal', 'elevated', 'hypertension_stage1', 'hypertension_stage2', 'hypertensive_crisis'],
    'hrv_sdnn_category':       ['low', 'normal', 'high'],
    'hrv_rmssd_category':      ['low', 'normal', 'high'],
    'hrv_pnn50_category':      ['low', 'normal', 'high'],
    'af_label':                ['non_af', 'af'],
    'arrhythmia_category':     ['sinus_rhythm', 'pvc', 'pac', 'vt', 'svt', 'unknown_0'],
    'spo2_category':           ['normal', 'abnormal'],
    'rr_category':             ['bradypnea', 'normal', 'tachypnea'],
    'sqi_category':            ['good_quality', 'noisy'],
    'stress_label':            ['baseline', 'stress', 'amusement', 'meditation'],
    'sdb_label':               ['normal_ahi<5', 'mild_5<=ahi<15', 'moderate_15<=ahi<30', 'severe_ahi>=30'],
}

cat_order_fig = [
    'heart_rate_category', 'blood_pressure_category',
    'hrv_sdnn_category', 'hrv_rmssd_category', 'hrv_pnn50_category',
    'af_label', 'arrhythmia_category',
    'spo2_category', 'rr_category', 'sqi_category',
    'stress_label', 'sdb_label',
] 

cat_display_fig = {
    'heart_rate_category':     'Heart Rate',
    'blood_pressure_category': 'Blood Pressure',
    'hrv_sdnn_category':       'HRV-SDNN',
    'hrv_rmssd_category':      'HRV-RMSSD',
    'hrv_pnn50_category':      'HRV-pNN50',
    'af_label':                'AF Detection',
    'arrhythmia_category':     'Arrhythmia',
    'spo2_category':           'SpO2',
    'rr_category':             'Respiratory Rate',
    'sqi_category':            'Signal Quality',
    'stress_label':            'Stress',
    'sdb_label':               'Sleep Apnea',
}

cat_order_fig = [c for c in cat_order_fig if c in _all_cat_answers]

fig, axes = plt.subplots(3, 4, figsize=(10, 7))
axes_flat = axes.flatten()

for idx, cat in enumerate(cat_order_fig):
    ax = axes_flat[idx]
    counter = _all_cat_answers[cat]

    labels_ordered = [l for l in LABEL_ORDER.get(cat, []) if l in counter]
    labels_ordered += [l for l in sorted(counter.keys()) if l not in labels_ordered]

    counts = [counter[l] for l in labels_ordered]
    display_labels = [ANSWER_DISPLAY.get(l, l) for l in labels_ordered]
    colors = CAT_PALETTES.get(cat, ['#B0B0B0'] * len(labels_ordered))
    if len(colors) < len(labels_ordered):
        colors = colors + ['#D0D0D0'] * (len(labels_ordered) - len(colors))

    bars = ax.bar(range(len(labels_ordered)), counts,
                  color=colors[:len(labels_ordered)], edgecolor='white', linewidth=0.3)

    total = sum(counts)
    for bar, count in zip(bars, counts):
        pct = count / total * 100
        y_off = bar.get_height() + total * 0.01
        ax.text(bar.get_x() + bar.get_width() / 2, y_off,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=6, color='#555555')

    ax.set_xticks(range(len(labels_ordered)))
    ax.set_xticklabels(display_labels, rotation=40, ha='right', fontsize=6.5)
    ax.set_title(cat_display_fig[cat], fontsize=9, fontweight='bold', pad=4)
    ax.set_ylabel('Count', fontsize=6.5)
    ax.tick_params(labelsize=7)
    ax.ticklabel_format(axis='y', style='scientific', scilimits=(0, 3))
    ax.yaxis.get_offset_text().set_fontsize(6)

for ax in axes_flat[len(cat_order_fig):]:
    ax.set_visible(False)

plt.tight_layout(h_pad=1.5, w_pad=1.0)
savefig('label_distributions_in_test_sets.pdf')
plt.show()
print('Saved: assets/figs/label_distributions_in_test_sets.pdf')

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(28, 14))

for ax, name in zip(axes.flat, dataset_names):
    ds = raw[name][SPLIT]
    ex = ds[3]
    sig = np.array(ex["signal"], dtype=np.float32)
    ax.set_title(name, fontsize=24, fontweight="bold")
    ax.plot(sig[:1250], color="#555555", linewidth=0.8)
    # ax.set_xlabel("sample", fontsize=7)
    ax.set_ylabel("amplitude", fontsize=7)
    ax.tick_params(labelsize=6)

    qa = parse_qa(ex["qa"])
    text_ctx = (ex["text"] or "").strip()
    qa_lines = []
    if isinstance(qa, dict):
        for cat, payload in list(qa.items())[:4]:
            ans = payload.get("answer", payload) if isinstance(payload, dict) else payload
            qa_lines.append(f"{cat}: {ans}")

plt.tight_layout()
savefig("ppg_grid.pdf")
plt.show()

In [ ]:
summary_rows = []
for name in dataset_names:
    cats_present = sorted(per_dataset_cats[name].keys())
    total_qa = sum(per_dataset_cats[name].values())
    summary_rows.append({
        "dataset": name,
        "train": len(raw[name]["train"]),
        "val": len(raw[name]["validation"]),
        "test": len(raw[name]["test"]),
        "sig_len": len(raw[name][SPLIT][0]["signal"]),
        "n_qa_cats": len(cats_present),
        "total_qa_pairs": total_qa,
        "qa_categories": ", ".join(cats_present),
    })

df_summary = pd.DataFrame(summary_rows).set_index("dataset")
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 250)
print(df_summary.to_string())
df_summary.to_csv(os.path.join(FIG_DIR, "dataset_summary.csv"))
